# nb_02_bronze_to_silver
**SYNTHETIC DATA · for training only · no real customer data**

Builds the conformed **silver** layer in `lh_silver` (Delta tables, overwrite):

| silver table | source | notes |
|---|---|---|
| `customer` | `lh_bronze/Tables/customers` (shortcut) | trimmed, email lower-cased; keeps `sin`, `date_of_birth` (protected later with OneLake security CLS) |
| `account` | `lh_bronze/Tables/accounts` (shortcut) | + `province` from customer (so every silver table can be filtered by province) |
| `loan` | `lh_bronze/Tables/loan_applications_raw` | de-duplicated on `application_id`, + `province`, `submitted_date` |
| `transaction_monthly` | `sqldb_core_banking/Tables/dbo/transactions` (direct abfss path) | monthly aggregate per account |

**Design choice - transactions:** read with a direct OneLake `abfss://` path to the SQL database's replicated table instead of a third shortcut. This keeps `lh_bronze` showing exactly the two shortcuts used on the slide, and demonstrates that any Fabric engine can read OneLake data in place when the caller has permission on the source item. (To use a shortcut instead, add `"transactions"` to `shortcut_tables` in nb_01 and switch `tx_path` below.)

Run by: pipeline `pl_daily_load` (second activity) or manually by a Contributor.

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# nb_02_bronze_to_silver - parameters
workspace_name = ""                 # "" = current workspace
bronze_name    = "lh_bronze"
silver_name    = "lh_silver"
sqldb_name     = "sqldb_core_banking"

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data

# Resolve workspace and item ids with semantic-link (sempy) so this notebook does
# NOT depend on a default lakehouse. All paths below are absolute abfss:// URLs.
import sempy.fabric as fabric
import notebookutils

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def resolve_ws(name):
    """Workspace id for a name; '' means the workspace this notebook runs in."""
    if not name:
        return fabric.get_workspace_id()
    try:
        return fabric.resolve_workspace_id(name)
    except AttributeError:  # older semantic-link versions
        df = fabric.list_workspaces()
        return df[df["Name"] == name]["Id"].iloc[0]

def resolve_item(name, item_type, ws_id):
    """Item id for a display name + type (Lakehouse, SQLDatabase, ...)."""
    try:
        return fabric.resolve_item_id(name, type=item_type, workspace=ws_id)
    except Exception:
        df = fabric.list_items(type=item_type, workspace=ws_id)
        hit = df[df["Display Name"] == name]
        if hit.empty:
            raise ValueError(f"{item_type} '{name}' not found in workspace {ws_id}")
        return hit["Id"].iloc[0]

def item_root(ws_id, item_id):
    return f"abfss://{ws_id}@{ONELAKE}/{item_id}"

ws_id     = resolve_ws(workspace_name)
bronze_id = resolve_item(bronze_name, "Lakehouse", ws_id)
silver_id = resolve_item(silver_name, "Lakehouse", ws_id)
sqldb_id  = resolve_item(sqldb_name, "SQLDatabase", ws_id)

bronze_root = item_root(ws_id, bronze_id)
silver_root = item_root(ws_id, silver_id)
sqldb_root  = item_root(ws_id, sqldb_id)

tx_path = f"{sqldb_root}/Tables/dbo/transactions"          # direct path (see design choice above)
# tx_path = f"{bronze_root}/Tables/transactions"           # alternative: third shortcut
print(f"[step] bronze={bronze_id} silver={silver_id} sqldb={sqldb_id}")

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
from pyspark.sql import functions as F, Window

customers_b    = spark.read.format("delta").load(f"{bronze_root}/Tables/customers")
accounts_b     = spark.read.format("delta").load(f"{bronze_root}/Tables/accounts")
loans_raw      = spark.read.format("delta").load(f"{bronze_root}/Tables/loan_applications_raw")
transactions_b = spark.read.format("delta").load(tx_path)

# ---- customer
customer = customers_b.select(
    F.col("customer_id").cast("int").alias("customer_id"),
    F.trim("first_name").alias("first_name"),
    F.trim("last_name").alias("last_name"),
    F.lower(F.trim("email")).alias("email"),
    F.trim("sin").alias("sin"),
    F.col("date_of_birth").cast("date").alias("date_of_birth"),
    F.upper(F.trim("province")).alias("province"),
    F.trim("segment").alias("segment"),
    F.col("created_at").cast("timestamp").alias("created_at"),
)
cust_prov = customer.select("customer_id", "province")

# ---- account (+ province)
account = (accounts_b
    .select(F.col("account_id").cast("int").alias("account_id"),
            F.col("customer_id").cast("int").alias("customer_id"),
            F.trim("account_type").alias("account_type"),
            F.trim("status").alias("status"),
            F.col("opened_at").cast("date").alias("opened_at"),
            F.col("balance").cast("decimal(18,2)").alias("balance"),
            F.trim("currency").alias("currency"))
    .join(cust_prov, "customer_id", "left"))

# ---- loan (latest ingest wins, + province)
latest = Window.partitionBy("application_id").orderBy(F.col("ingest_ts").desc())
loan = (loans_raw
    .withColumn("rn", F.row_number().over(latest)).filter("rn = 1").drop("rn")
    .join(cust_prov, "customer_id", "left")
    .select("application_id", "customer_id", "province", "product",
            F.col("amount").cast("decimal(18,2)").alias("amount"),
            F.col("rate").cast("decimal(6,3)").alias("rate"),
            "status", "submitted_at",
            F.to_date("submitted_at").alias("submitted_date"),
            "source_file"))

# ---- transaction_monthly
acct_keys = account.select("account_id", "customer_id", "province")
transaction_monthly = (transactions_b
    .join(acct_keys, "account_id", "inner")
    .withColumn("month_start", F.trunc("txn_ts", "month"))
    .groupBy("month_start", "account_id", "customer_id", "province")
    .agg(F.count(F.lit(1)).cast("int").alias("txn_count"),
         F.sum(F.when(F.col("direction") == "DEBIT",  F.col("amount")).otherwise(F.lit(0))).cast("decimal(18,2)").alias("debit_total"),
         F.sum(F.when(F.col("direction") == "CREDIT", F.col("amount")).otherwise(F.lit(0))).cast("decimal(18,2)").alias("credit_total"))
    .withColumn("net_amount", (F.col("credit_total") - F.col("debit_total")).cast("decimal(18,2)")))

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
def write_silver(df, name):
    path = f"{silver_root}/Tables/{name}"
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .save(path))
    n = spark.read.format("delta").load(path).count()
    print(f"[step] lh_silver.{name:<20} {n:>8,} rows")
    return n

counts = {
    "customer":            write_silver(customer, "customer"),
    "account":             write_silver(account, "account"),
    "loan":                write_silver(loan, "loan"),
    "transaction_monthly": write_silver(transaction_monthly, "transaction_monthly"),
}
print(counts)
display(spark.read.format("delta").load(f"{silver_root}/Tables/customer")
        .groupBy("province").count().orderBy(F.desc("count")))